In [ ]:
# Hosted D2L setup: fetch the exact helper module used to build this notebook.
from pathlib import Path
from urllib.request import urlretrieve
from importlib.metadata import PackageNotFoundError, version
import importlib.util, os, subprocess, sys

required = ['numpy', 'pandas', 'matplotlib', 'requests', 'scipy', 'pillow', 'regex', 'tensorflow', 'keras', 'protobuf', 'ml-dtypes']
imports = {'pillow': 'PIL', 'protobuf': 'google.protobuf', 'ml-dtypes': 'ml_dtypes'}
pinned = {}
fallbacks = {'tensorflow': 'tensorflow==2.21.0', 'keras': 'keras==3.14.0', 'protobuf': 'protobuf==7.34.1', 'ml-dtypes': 'ml-dtypes==0.5.4'}
device = os.environ.get("D2L_HOSTED_DEVICE", "auto").lower()
if device not in ("auto", "cpu", "gpu"):
    raise ValueError(f"Invalid D2L_HOSTED_DEVICE={device!r}")
if device == "auto":
    try:
        gpu = (Path("/dev/nvidia0").exists() or
               subprocess.run(["nvidia-smi", "-L"], capture_output=True,
                              timeout=5).returncode == 0)
    except (FileNotFoundError, subprocess.SubprocessError):
        gpu = False
else:
    gpu = device == "gpu"
if not gpu:
    os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")
    os.environ.setdefault("JAX_PLATFORMS", "cpu")
tensorflow_version = None
if 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_version = version("tensorflow")
    except PackageNotFoundError:
        pass
# Colab's CPU image currently carries a CUDA-enabled TensorFlow wheel. Its
# first ordinary tensor operation probes CUDA and emits an error-level cuInit
# diagnostic. JAX notebooks also use TensorFlow for data loading, so overlay
# the matching CPU build in both CPU variants. Keep the provider's
# ``tensorflow`` distribution metadata: other preinstalled Colab packages
# depend on that distribution name, while both wheels expose the same module.
if not gpu and 'tensorflow' in ("tensorflow", "jax"):
    try:
        tensorflow_cpu_version = version("tensorflow-cpu")
    except PackageNotFoundError:
        tensorflow_cpu_version = None
    if (tensorflow_version is not None and
            tensorflow_cpu_version != tensorflow_version):
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q", "--no-deps",
            f"tensorflow-cpu=={tensorflow_version}",
        ])
if "tf-keras" in fallbacks and tensorflow_version is not None:
    fallbacks["tf-keras"] = f"tf-keras=={tensorflow_version}"
missing = []
for package in required:
    if package in pinned:
        wanted, cpu_requirement, gpu_requirement, match = pinned[package]
        requirement = gpu_requirement if gpu else cpu_requirement
        if requirement is None:  # a GPU-only package on a CPU runtime
            continue
        try:
            installed = version(package)
        except PackageNotFoundError:
            installed = None
        actual = (installed.split("+", 1)[0]
                  if installed is not None and match == "public" else installed)
        if actual != wanted:
            missing.append(requirement)
    elif importlib.util.find_spec(imports.get(package, package)) is None:
        missing.append(fallbacks.get(package, package))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

mismatched = []
for package, (wanted, cpu_requirement, _, match) in pinned.items():
    if not gpu and cpu_requirement is None:
        continue
    try:
        installed = version(package)
    except PackageNotFoundError:
        installed = None
    actual = (installed.split("+", 1)[0]
              if installed is not None and match == "public" else installed)
    if actual != wanted:
        mismatched.append(f"{package}={installed!r} (expected {wanted})")
if mismatched:
    raise RuntimeError("Hosted runtime setup failed: " + ", ".join(mismatched))

root = Path(".d2l-hosted") / "46d487bb9b699b97c7c5fc5edd866e4d2ad7919b"
package = root / "d2l"
package.mkdir(parents=True, exist_ok=True)
base = "https://raw.githubusercontent.com/smolix/d2l-neu/46d487bb9b699b97c7c5fc5edd866e4d2ad7919b/d2l"
for name in ('__init__.py', 'tensorflow.py'):
    target = package / name
    if not target.exists():
        urlretrieve(f"{base}/{name}", target)
if str(root.resolve()) not in sys.path:
    sys.path.insert(0, str(root.resolve()))
pythonpath = os.environ.get("PYTHONPATH", "").split(os.pathsep)
if str(root.resolve()) not in pythonpath:
    os.environ["PYTHONPATH"] = os.pathsep.join(
        [str(root.resolve()), *[entry for entry in pythonpath if entry]]
    )


# Latent Variables, EM, and the ELBO

Every likelihood we met in that section was a clean
product of factors, one per observation, so taking logarithms turned it into a
sum we could differentiate term by term. That convenience rested on a quiet
assumption: that we get to observe everything the model generates.

In practice, we often miss a crucial piece of the data. Imagine collecting
$500$ measurements drawn from two distinct populations, $300$ from one and
$200$ from the other, only to discover that nobody recorded which population
produced which measurement. If those labels had survived, fitting the model
would take one line per population: a mixing fraction, a mean, and a variance.
Without them, the probability of each measurement becomes a sum over the two
populations it might have come from. Unfortunately, the logarithm of a sum
does not split apart; every parameter ends up entangled in every term, and
setting the gradient to zero leaves equations with no closed-form solution.
The same obstacle appears whenever a model explains observations through
hidden structure: the cluster of a customer, the topic of a document, or the
random code a generator turns into an image. Such unobserved quantities are
called *latent variables*.

The central idea of this section is simple and surprisingly effective: guess
what we cannot see using probabilities rather than hard labels, and then fit
the model as if those soft guesses were real data. What makes this idea work
is that our guess does not even have to be right. For *any* probabilistic
guess, the resulting easy objective lies below the true log-likelihood, and it
touches the log-likelihood precisely when the guess equals the posterior
distribution over the hidden labels. This lower bound is the key result to
remember. Expectation--maximization climbs it directly, while variational
inference and variational autoencoders climb it approximately whenever the
exact posterior is out of reach.

We will develop this one idea from start to finish: first setting up mixture
models, then deriving the lower bound and measuring its gap, and finally
running the resulting algorithm on our $500$ points, once from a starting point
where it recovers both populations and once from a symmetric start where it
gets stuck.

In [1]:
import math
import numpy as np
import tensorflow as tf
from d2l import tensorflow as d2l

## Mixture Models

Let us translate this missing-label puzzle into symbols. A probabilistic model
with a latent variable $z$ specifies a joint distribution over the hidden and
observed quantities,

$$
p(x, z\mid\boldsymbol{\theta}) = p(z\mid\boldsymbol{\theta})\, p(x \mid z, \boldsymbol{\theta}),
$$

of which we observe only $x$. To compute the likelihood of what we actually
saw, we must sum out all possible values of the hidden variable (or integrate
them out when $z$ is continuous), giving the *marginal* likelihood:

$$
p(x\mid\boldsymbol{\theta}) = \sum_{z} p(z\mid\boldsymbol{\theta})\, p(x\mid z, \boldsymbol{\theta}).
$$

In words, the model's probability for $x$ adds up every way the model could
have generated $x$, one term for each value of the hidden variable.

The canonical example is the **Gaussian mixture model** (GMM). Here the latent
variable $z \in \{1,\ldots,K\}$ selects component $k$ with mixing probability
$\pi_k$, and the observation is then drawn from that component's Gaussian,
$x \mid z{=}k \sim \mathcal{N}(\mu_k, \sigma_k^2)$. Putting the $n$
observations together, the negative log-likelihood (NLL) is

$$
\ell(\boldsymbol{\theta})
  = -\sum_{i=1}^n \log\Big(\sum_{k=1}^K \pi_k\, \mathcal{N}(x_i;\, \mu_k, \sigma_k^2)\Big),
\qquad \boldsymbol{\theta} = (\pi_k, \mu_k, \sigma_k)_{k=1}^K.
$$

All the trouble comes from the sum sitting *inside* the logarithm. The
logarithm stops at the mixture and never reaches the individual Gaussian
densities, where it would otherwise cancel the exponential and leave a simple
quadratic. Compare this with the *complete-data* problem: if an oracle handed
us every label $z_i$, the likelihood would separate cleanly by component, and
maximum likelihood would reduce to estimators we already know. Each mixing
weight $\pi_k$ would simply be the fraction of points belonging to component
$k$, just like the coin-bias estimator $\hat\theta = n_H/n$ in
that section, while $\mu_k$ and $\sigma_k^2$ would be
the ordinary sample mean and variance of those points. Because the problem is
hard only when the labels are missing, a natural strategy suggests itself:
estimate the missing labels as probabilities, and then solve the easy
complete-data problem.

What does such a probabilistic guess look like for a single point? Consider
one observation at $x = 0.5$ under two equally weighted components,
$\mathcal{N}(-1, 1)$ and $\mathcal{N}(1, 1)$. Their densities at $x = 0.5$ are
proportional to $e^{-(1.5)^2/2} = e^{-1.125}$ and
$e^{-(0.5)^2/2} = e^{-0.125}$, so Bayes' rule assigns the left component the
posterior probability

$$
\frac{e^{-1.125}}{e^{-1.125}+e^{-0.125}} = \frac{1}{1+e} = 0.27.
$$

Rather than forcing a hard choice, Bayes' rule splits the point $27\%$ to
$73\%$ between the two components. Move the observation far to the right, say
to $x = 3$, and the posterior becomes $99.8\%$ certain it belongs to the right
component; place it at the midpoint $x = 0$, and the split is an even
$50\%$--$50\%$. The soft guess is confident where the data are clear and
appropriately undecided where the components overlap
(the figure).

![Soft assignment as a ratio of density heights, for two equally weighted components $\mathcal{N}(-1,1)$ and $\mathcal{N}(1,1)$. (a) At $x=0.5$ the densities have heights $h_1$ and $h_2$, and the left component receives the share $r_1=h_1/(h_1+h_2)=0.27$. (b) The two posterior probabilities $r_1(x)$ and $r_2(x)$: an even split at $x=0$, $0.27$ to $0.73$ at $x=0.5$, and $0.998$ for the right component at $x=3$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-soft-resp.svg)

## The Evidence Lower Bound

To turn this intuitive strategy into an algorithm with a mathematical
guarantee, we construct a lower bound on the log-likelihood in which the
logarithm moves back inside the sum. Let $q(z)$ be *any* probability
distribution over the latent variable, representing our soft guess. We
multiply and divide inside the marginal likelihood by $q(z)$ and apply
Jensen's inequality: because the logarithm is concave,
$\log \mathbb{E}[\cdot] \ge \mathbb{E}[\log(\cdot)]$
(that section). This yields

$$
\log p(x\mid\boldsymbol{\theta})
  = \log \sum_z q(z)\, \frac{p(x,z\mid\boldsymbol{\theta})}{q(z)}
  \;\ge\; \sum_z q(z) \log \frac{p(x,z\mid\boldsymbol{\theta})}{q(z)}
  \;=:\; \mathcal{L}(q, \boldsymbol{\theta}).
$$

Here the sums run over all $z$ with $q(z) > 0$. If $q$ happens to put zero
probability on some $z$ where the joint distribution is positive, the first
equality simply becomes $\ge$, so the bound still holds. Notice that the
marginal $p(x\mid\boldsymbol{\theta})$ plays the same role for the latent
variable $z$ that the evidence $P(X)$ (the denominator of Bayes' rule in
that section) plays for model parameters: it is the probability of
the observed data after summing out everything unobserved. For that reason it
is also called the *evidence*, and the right-hand side
$\mathcal{L}(q, \boldsymbol{\theta})$ is called the **evidence lower bound**
(**ELBO**). Splitting the logarithm of the ratio separates the ELBO into two
transparent pieces:

$$
\mathcal{L}(q,\boldsymbol{\theta})
  = \mathbb{E}_{q}\big[\log p(x,z\mid\boldsymbol{\theta})\big] + H(q).
$$

The first term is the expected *complete-data* log-likelihood, our easy
objective averaged over the soft guess $q$. The second term is the entropy of
the guess, which does not depend on $\boldsymbol{\theta}$ at all. Even better,
we can measure the exact gap between the ELBO and the true log-likelihood: it
is a KL divergence, and both EM and variational inference rest on this single
identity.

**Proposition (the ELBO gap is a KL divergence).** *For every
$\boldsymbol{\theta}$ and every distribution $q$ that puts no mass outside the
support of the posterior $p(z\mid x,\boldsymbol{\theta})$,*

$$
\log p(x\mid\boldsymbol{\theta})
  = \mathcal{L}(q,\boldsymbol{\theta})
  + D_{\textrm{KL}}\big(q(z)\,\|\,p(z\mid x,\boldsymbol{\theta})\big).
$$

*Hence $\mathcal{L}(q,\boldsymbol{\theta}) \le \log p(x\mid\boldsymbol{\theta})$,
with equality if and only if $q$ equals the posterior
$p(z\mid x,\boldsymbol{\theta})$.*

**Proof.** Substitute Bayes' rule,
$p(z\mid x,\boldsymbol{\theta}) = p(x,z\mid\boldsymbol{\theta})/p(x\mid\boldsymbol{\theta})$,
directly into the definition of the KL divergence:

$$
D_{\textrm{KL}}\big(q \,\|\, p(z\mid x,\boldsymbol{\theta})\big)
  = \mathbb{E}_q\!\left[\log \frac{q(z)\, p(x\mid\boldsymbol{\theta})}{p(x,z\mid\boldsymbol{\theta})}\right]
  = \log p(x\mid\boldsymbol{\theta}) - \mathcal{L}(q,\boldsymbol{\theta}),
$$

where $\log p(x\mid\boldsymbol{\theta})$ pulls straight out of the expectation
because it does not depend on $z$. Rearranging gives
the equation. By Gibbs' inequality
(that section), the KL divergence is always nonnegative and
vanishes if and only if its two arguments match. $\blacksquare$

Let us check this identity on our concrete example $x = 0.5$ from the previous
subsection. There the true log-evidence is $\log p(x) = -1.42$. If we use the
undecided guess $q = (\tfrac{1}{2}, \tfrac{1}{2})$, the ELBO evaluates to
$-1.54$, and the missing $0.12$ nats are the exact KL divergence from
$(\tfrac{1}{2}, \tfrac{1}{2})$ to the posterior $(0.27, 0.73)$. In other
words, the bound falls short of the log-evidence by the mismatch between our
guess and the true posterior, and by nothing else. the figure
illustrates how this lower bound behaves as $\boldsymbol{\theta}$ varies.

![The evidence and its lower bound. For a fixed guess $q$, the ELBO $\mathcal{L}(q,\theta)$ lies below the evidence $\log p(x\mid\theta)$ at every $\theta$, and the vertical gap between the curves is the KL divergence from $q$ to the posterior $p(z\mid x,\theta)$. In EM, the E-step sets $q$ to the posterior at the current iterate $\theta^{(t)}$, closing the gap so the bound touches the evidence there; the M-step then climbs the bound to its peak $\theta^{(t+1)}$. Because the bound touches the evidence at $\theta^{(t)}$ and never rises above it, the evidence at $\theta^{(t+1)}$ is guaranteed to be at least as high as at $\theta^{(t)}$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-elbo.svg)

## Expectation--Maximization

The ELBO identity turns our difficult optimization problem into **coordinate
ascent** on $\mathcal{L}(q, \boldsymbol{\theta})$: maximize with respect to
one argument while holding the other fixed, swap roles, and repeat. Given a
dataset $X = \{x_1,\ldots,x_n\}$ of independent observations, the bound
applies to each observation separately and sums across the dataset, with one
guess $q_i$ per data point. The **expectation--maximization** (EM) algorithm
alternates between these two coordinates [@Dempster.Laird.Rubin.1977]:

* **E-step.** Hold $\boldsymbol{\theta}^{(t)}$ fixed and maximize over the
  guesses $q_i$. By the equality condition in the equation, the
  optimal choice for every point is its exact posterior,
  $q_i^{(t)}(z) = p(z \mid x_i, \boldsymbol{\theta}^{(t)})$. This drives every
  KL gap to zero and makes the bound *tight* at the current parameters:
  $\sum_i \mathcal{L}(q_i^{(t)}, \boldsymbol{\theta}^{(t)}) = \log P(X\mid\boldsymbol{\theta}^{(t)})$.
* **M-step.** Hold the guesses $q_i^{(t)}$ fixed and maximize over the
  parameters $\boldsymbol{\theta}$:

  $$
  \boldsymbol{\theta}^{(t+1)} = \mathop{\mathrm{argmax}}_{\boldsymbol{\theta}} \sum_i \mathbb{E}_{q_i^{(t)}}\big[\log p(x_i, z\mid\boldsymbol{\theta})\big],
  $$

  where the entropy terms $H(q_i^{(t)})$ drop out because they do not depend
  on $\boldsymbol{\theta}$. We are back to the easy complete-data objective,
  with soft probabilities standing in for the missing oracle labels.

Why can we trust this alternating procedure? Because the geometry in
the figure guarantees that every single step improves (or at
worst preserves) the true log-likelihood.

**Proposition (EM never decreases the likelihood).** *Every EM iteration
satisfies $\log P(X\mid\boldsymbol{\theta}^{(t+1)}) \ge \log P(X\mid\boldsymbol{\theta}^{(t)})$.*

**Proof.** Chain together three one-line facts:

$$
\log P(X\mid\boldsymbol{\theta}^{(t+1)})
  \;\ge\; \sum_i \mathcal{L}\big(q_i^{(t)}, \boldsymbol{\theta}^{(t+1)}\big)
  \;\ge\; \sum_i \mathcal{L}\big(q_i^{(t)}, \boldsymbol{\theta}^{(t)}\big)
  \;=\; \log P(X\mid\boldsymbol{\theta}^{(t)}).
$$

The first inequality is the ELBO bound the equation evaluated at the
new parameters, the second holds because the M-step maximizes the bound over
$\boldsymbol{\theta}$, and the final equality is the tightness secured by the
E-step. $\blacksquare$

For our Gaussian mixture model, both steps come out in closed form. The E-step
computes the **responsibilities**, the posterior probabilities that point $i$
was generated by component $k$, just as we did by hand for $x = 0.5$:

$$
r_{ik} = \frac{\pi_k\, \mathcal{N}(x_i;\, \mu_k, \sigma_k^2)}
              {\sum_{j} \pi_j\, \mathcal{N}(x_i;\, \mu_j, \sigma_j^2)}.
$$

The M-step then refits each component using responsibility-weighted counting
and averaging. Writing $n_k = \sum_i r_{ik}$ for the effective number of
points assigned to component $k$, we obtain

$$
\pi_k = \frac{n_k}{n}, \qquad
\mu_k = \frac{1}{n_k}\sum_{i} r_{ik}\, x_i, \qquad
\sigma_k^2 = \frac{1}{n_k}\sum_{i} r_{ik}\,(x_i - \mu_k)^2.
$$

Notice how familiar these formulas are: they are the ordinary complete-data
estimators (mixing fraction, sample mean, and sample variance), simply softened.
Instead of casting one hard vote for a single component, each data point
divides its vote across the components according to its responsibilities. In
fact, if we force all components to share a single fixed variance $\sigma^2$
and shrink $\sigma^2 \to 0$, the responsibilities harden into $0/1$
assignments to the nearest mean, and EM reduces directly to the K-means
algorithm.

The entire algorithm takes only a dozen lines of code. Let us generate $300$
points from $\mathcal{N}(-2, 0.8^2)$ and $200$ from $\mathcal{N}(1.5, 0.6^2)$
and write a short function `em` for a one-dimensional mixture of $K$ Gaussians
that records the log-likelihood at every step alongside the final fitted
parameters.

In [2]:
rng = np.random.default_rng()
x = np.concatenate([rng.normal(-2.0, 0.8, 300), rng.normal(1.5, 0.6, 200)])

def gauss(x, mu, sigma):
    return np.exp(-0.5 * ((x - mu) / sigma)**2) / (sigma * np.sqrt(2 * np.pi))

def em(x, pi, mu, sigma, steps):
    """EM for a 1-D Gaussian mixture: log-likelihood trace and final fit."""
    pi, mu, sigma = (np.array(v, dtype=float) for v in (pi, mu, sigma))
    lls = []
    for _ in range(steps):
        joint = pi * gauss(x[:, None], mu, sigma)  # (n, K)
        lls.append(np.log(joint.sum(axis=1)).sum())
        r = joint / joint.sum(axis=1, keepdims=True)  # E-step
        n_k = r.sum(axis=0)  # M-step
        pi, mu = n_k / len(x), (r * x[:, None]).sum(axis=0) / n_k
        sigma = np.sqrt((r * (x[:, None] - mu)**2).sum(axis=0) / n_k)
    return np.array(lls), pi, mu, sigma

Our proposition guarantees that the log-likelihood will never decrease, but it
says nothing about how quickly EM climbs. Let us test it from a rather poor
initial guess, two heavily overlapping unit-variance components centered at
$\pm 0.5$, run $30$ iterations, and inspect the first six log-likelihood values
alongside the final step.

In [3]:
lls, pi, mu, sigma = em(x, [0.5, 0.5], [-0.5, 0.5], [1, 1], 30)
print('steps 0-5:', lls[:6].round(1), ' step 29:', lls[-1].round(1))
print(f'pi = {pi.round(2)}, mu = {mu.round(2)}, sigma = {sigma.round(2)}')

steps 0-5: [-1271.8  -985.3  -945.7  -878.5  -845.1  -842.3]  step 29: -842.2
pi = [0.6 0.4], mu = [-2.01  1.5 ], sigma = [0.75 0.57]


The log-likelihood climbs by roughly $400$ nats, rising from around
$-1{,}280$ to around $-860$ (both figures vary by a few tens of nats with the
random sample). Remarkably, almost the entire gain happens immediately: after
just six steps the log-likelihood is already within about a nat of its final
value, and the recovered weights, means, and scales land within about $0.2$ of
the true generating parameters $(0.6, 0.4)$, $(-2, 1.5)$, and $(0.8, 0.6)$.

Despite this clean convergence, EM comes with four limitations worth knowing.
The first is harmless: permuting the labels of the $K$ components produces
$K!$ identical copies of every maximum. This *label switching* is the same
non-identifiability we encounter when permuting the hidden units of a neural
network, and it does not make any solution worse than another.

The second limitation is more consequential: EM stops at the first stationary
point it hits, whether or not that point is a global maximum. A symmetric
initialization illustrates this sharply. Before running the next cell, pause
and predict what EM will do if both components start with the exact same mean
and scale.

In [4]:
lls2, pi2, mu2, sigma2 = em(x, [0.5, 0.5], [0, 0], [1, 1], 30)
print(f'log-likelihood {lls2[0]:.1f} -> {lls2[-1]:.1f}, '
      f'{lls[-1] - lls2[-1]:.1f} below the previous fit')
print(f'pi = {pi2.round(2)}, mu = {mu2.round(2)}, sigma = {sigma2.round(2)}')
print(f'mean of x = {x.mean():.2f}, standard deviation of x = {x.std():.2f}')

log-likelihood -1405.6 -> -1017.8, 175.7 below the previous fit
pi = [0.5 0.5], mu = [-0.59 -0.59], sigma = [1.85 1.85]
mean of x = -0.59, standard deviation of x = 1.85


The two components never break apart. Because they begin identically, they
receive identical responsibilities on every single data point, so the M-step
keeps them locked together. EM settles into the single-Gaussian fit, just the
overall mean and standard deviation of the entire dataset, more than $100$ nats
below the two-cluster solution. Every fixed point of EM is a stationary point
of the likelihood, not necessarily a maximum, which is why practitioners
restart EM from several random initializations. Random initialization breaks
this exact same symmetry among hidden units in a neural network layer
(that section): units that start equal receive equal
gradients and remain equal forever.

The third limitation is a quirk of continuous mixtures: the likelihood itself
is unbounded. Place the second component directly on top of the first data
point, $\mu_2 = x_1$, with weight $\pi_2 = 1/n$ and standard deviation
$\epsilon$. Its density at $x_1$ is $\tfrac{1}{n}(2\pi)^{-1/2}\epsilon^{-1}$,
while the first component continues to cover the remaining $n - 1$ points,
giving

$$
\ell(\boldsymbol{\theta}) \le \log\big(n\sqrt{2\pi}\big) + \log\epsilon + \textrm{const},
$$

where the constant gathers the contribution of the other $n - 1$ points. As
$\epsilon \to 0$, the negative log-likelihood plunges to $-\infty$: the global
maximum-likelihood "solution" is simply an infinitely sharp spike perched on a
single data point (the figure). Imposing a lower floor on the variances, or placing a prior
over them, eliminates these degenerate spikes. Once the set of parameters at
least as good as the starting point is compact, every limit point of EM is
guaranteed to be a stationary point of the regularized problem
[@Wu.1983].

![The degenerate maximum of a Gaussian mixture likelihood. The first component (blue) covers $n-1$ of the data points (dots). The second (orange) sits on the remaining point, $\mu_2=x_1$, with standard deviation $\sigma_2=\epsilon$. As $\epsilon$ shrinks from $0.3$ to $0.1$ to $0.03$, its density at $x_1$ grows like $1/\epsilon$, and the likelihood grows with it without bound.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/mdl-prob-gmm-spikes.svg)

The fourth limitation is what leads beyond EM altogether. Our monotonicity
proof depended on plugging the *exact* posterior into the E-step; whenever we
must settle for an approximate E-step, that guarantee disappears. A
*variational autoencoder* [@Kingma.Welling.2014], for example, keeps the
very same ELBO objective but replaces the exact per-point posterior with an
encoder neural network $q_{\boldsymbol{\phi}}(z\mid x)$ trained jointly with
the decoder, because a deep nonlinear generator has no closed-form posterior.
Instead of exact EM steps, it climbs the ELBO using stochastic gradients.

Here the bound outgrows the algorithm that introduced it. Nothing in the ELBO
requires the hidden variable $z$ to be a cluster label: if we put the model
parameters themselves in place of $z$, the posterior that the E-step asks for
is the posterior over parameters, the very object we were left wanting at the
end of that section. That posterior is rarely
available in closed form either, so that section
turns to the practical methods we use to approximate and sample from it.

## Summary

When a model includes hidden variables, its marginal likelihood traps a sum
inside a logarithm, turning what would have been simple counting and averaging
into an optimization problem with no closed form. Introducing a probabilistic
guess $q$ over the hidden variables restores the easy complete-data objective
as a lower bound: the ELBO falls below the log-evidence by exactly the KL
divergence from $q$ to the true latent posterior. EM alternates between closing
that gap (the E-step) and maximizing the resulting bound over the parameters
(the M-step). Although every iteration is guaranteed never to decrease the
likelihood, EM can stop at non-optimal stationary points, requires a variance
floor or prior to prevent degenerate spikes in continuous mixtures, and needs
an exact posterior to keep its monotonicity guarantee.

## Exercises

1. [code] **EM speed and missing information.** ● Consider the mixture
   $p(x\mid\pi)=\pi f_0(x)+(1-\pi)f_1(x)$ with known components
   $f_0=\mathcal N(\cdot\,;0,1)$ and $f_1=\mathcal N(\cdot\,;\Delta,1)$, in
   which only the mixing weight $\pi$ is unknown.
    1. Show that EM updates the weight by $\pi^{(t+1)}=M(\pi^{(t)})$ with
       $M(\pi)=\tfrac1n\sum_i r_i(\pi)$ and responsibilities
       $r_i(\pi)=\pi f_0(x_i)/p(x_i\mid\pi)$.
    1. Near a fixed point $\hat\pi=M(\hat\pi)$, each iteration multiplies the
       error by approximately $M'(\hat\pi)$. Define the observed information
       $I_{\textrm{obs}}=-\frac{d^2}{d\pi^2}\sum_i\log p(x_i\mid\pi)$ at
       $\hat\pi$, and the complete-data information
       $I_{\textrm{com}}=n/(\hat\pi(1-\hat\pi))$ that the labels would carry
       if they were observed (compare that section). Show
       that

       $$M'(\hat\pi)=1-\frac{I_{\textrm{obs}}}{I_{\textrm{com}}}.$$

       Express both sides in terms of the responsibilities and use
       $\sum_i r_i(\hat\pi)=n\hat\pi$. @Dempster.Laird.Rubin.1977
       prove the general form of this result: the convergence rate of EM is
       governed by the fraction of information hidden by the latent variables.
    1. Draw $n=10{,}000$ points with $\pi=0.3$ for each
       $\Delta\in\{0.5,1,2,4\}$, run EM from $\pi^{(0)}=0.5$, and compare the
       measured ratio of successive errors against the prediction from (b).
       Taking $\hat\pi$ from a long run, how many iterations does each
       $\Delta$ need to bring the error below $10^{-6}$?
    1. What does the ratio $I_{\textrm{obs}}/I_{\textrm{com}}$ imply about the
       standard error of $\hat\pi$, compared with a dataset of the same size
       whose labels were observed?

1. **The bound by hand.** Take the single observation $x=0.5$ and the two
   equally weighted components $\mathcal N(-1,1)$ and $\mathcal N(1,1)$ from
   the text.
    1. Compute $\log p(x)$, along with the ELBO the equation for the
       uniform guess $q=(\tfrac12,\tfrac12)$ and for the posterior
       $q=(0.27,0.73)$, verifying the values quoted in the text.
    1. Verify the equation for the uniform guess by computing the
       KL divergence directly.
    1. A hard guess puts all of $q$ on the right component. Compute its ELBO
       and its gap. An algorithm that alternates hard guesses with M-steps
       still never decreases the ELBO; why is its log-likelihood no longer
       guaranteed to increase?

1. **K-means as a limit.** Let all $K$ components of a Gaussian mixture share
   equal weights and a single fixed variance $\sigma^2$, so that only the means
   are fitted.
    1. Write the responsibilities $r_{ik}$ as a softmax of
       $-(x_i-\mu_k)^2/(2\sigma^2)$ over $k$. What quantity plays the role of
       the temperature? Show that as $\sigma^2\to0$, the responsibility of the
       nearest mean tends to $1$ whenever that nearest mean is unique.
    1. Show that the M-step the equation then sets each mean to
       the average of the points assigned to it, and that the two steps
       together never increase $\sum_i (x_i-\mu_{z_i})^2$, where $z_i$ is the
       component assigned to point $i$.
    1. In the symmetric start from the text, both means begin at the same
       point. What does K-means do from such a start, and what does this imply
       about how an implementation should break ties?

[Discussions](https://d2l.discourse.group/t/36881)